# **EmporiUm - Sales Analysis**

## Purpose


Assigned by the marketing manager at EmporiUm, my role is to interpret the company’s sales data to provide data‑driven insights that support  decision‑making. The purpose of this Jupyter notebook is to deliver a clear sales analysis for my regions, Colorado and Maine.

### Sales Territorys



> **State**
> > _Colorado_  
> > _Maine_
> > 

> **Territory Manager**  
> > *Jim Heck*  
> > *Erbayne Middleton*

> **Region**
> > *West*  
> > *Northeast*
> > 

In [179]:
import pandas as pd
import numpy as np
import math

## **Core Marketing Analysis**

### File Cleaning Function


In [180]:
def file_clean(file_var):
     file_var.columns = (file_var.columns.str.strip().str.lower().str.replace(" ", "_"))
     return file_var

### Opening the customer list file and giving the colounm names

In [181]:
customers = pd.read_csv("customer_list.csv", sep="|")
customers = file_clean(customers)
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 521 entries, 0 to 520
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   cust_id      521 non-null    int64
 1   date         521 non-null    str  
 2   time         521 non-null    str  
 3   name         521 non-null    str  
 4   email        521 non-null    str  
 5   phone        520 non-null    str  
 6   sms-opt-out  520 non-null    str  
dtypes: int64(1), str(6)
memory usage: 28.6 KB


### Opening the products file and giving the colounm names

In [182]:
products = pd.read_csv("Products.csv")
products = file_clean(products)
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 669 entries, 0 to 668
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   prod_num       669 non-null    str  
 1   product        669 non-null    str  
 2   categoryid     669 non-null    int64
 3   subcategoryid  669 non-null    str  
dtypes: int64(1), str(3)
memory usage: 21.0 KB


### Opening the Products Categories file and giving the colounm names

In [183]:
categories = pd.read_csv("ProductCategories.csv")
categories = file_clean(categories)
categories.info()

<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   categoryid     52 non-null     int64
 1   category       52 non-null     str  
 2   subcategoryid  52 non-null     str  
 3   subcategory    52 non-null     str  
dtypes: int64(1), str(3)
memory usage: 1.8 KB


### Opening the Store Detail file and giving the colounm names

In [184]:
store = pd.read_csv("StoreDetail.csv")
store = file_clean(store)
store.info()

<class 'pandas.DataFrame'>
RangeIndex: 111 entries, 0 to 110
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   store_location     111 non-null    str  
 1   state              111 non-null    str  
 2   store_id           111 non-null    int64
 3   territory_manager  111 non-null    str  
 4   region             111 non-null    str  
 5   region_director    111 non-null    str  
dtypes: int64(1), str(5)
memory usage: 5.3 KB


### Opening the Store Sales file and giving the colounm names

In [185]:
sales = pd.read_csv("StoreSales.csv")
sales = file_clean(sales)
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 335129 entries, 0 to 335128
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_date  335129 non-null  str    
 1   store_id          335129 non-null  int64  
 2   rewardsid         34943 non-null   float64
 3   prod_num          335129 non-null  str    
 4   sale_amount       335129 non-null  float64
dtypes: float64(2), int64(1), str(2)
memory usage: 12.8 MB


## Territory managers for Maine and Chicago

In [186]:
territory = ["Colorado","Maine"]
stores = store[store["state"].isin(territory)]
stores[["state", "territory_manager"]].drop_duplicates()

,state,territory_manager
0,Colorado,Jim Heck
37,Maine,Erbayne Middleton


## Store IDs and Cities for Maine and Colorado

In [187]:
stores[["state", "store_location", "store_id"]].sort_values(["state", "store_id"])

,state,store_location,store_id
0,Colorado,Aurora,701
1,Colorado,Berthoud,702
2,Colorado,Boulder,703
3,Colorado,Castle Rock,704
4,Colorado,Denver,705
5,Colorado,Englewood,706
6,Colorado,Fort Collins,707
7,Colorado,Grand Junction,708
8,Colorado,Greeley,709
9,Colorado,Lafayette,710


## Colorado Filtering

In [214]:
colorado_stores = store[store["state"] == "Colorado"]
colorado_id = colorado_stores["store_id"].unique()
colorado_sales = sales[sales["store_id"].isin(colorado_id)]

## Monthly total revenue for Colorado

In [215]:
# making the transaction date a readable format as well as changing the name to year_month for readablity
colorado_sales = colorado_sales.merge(colorado_stores[["store_id", "store_location"]], on="store_id", how="left")
colorado_sales["transaction_date"] = pd.to_datetime(colorado_sales["transaction_date"])
colorado_sales["year_month"] = colorado_sales["transaction_date"].dt.to_period("M")

# Adds the total revenue monthly
colorado_grouped_monthly = colorado_sales.groupby("year_month")
colorado_revenue_monthly = colorado_grouped_monthly["sale_amount"].sum().reset_index()
colorado_revenue_monthly = colorado_revenue_monthly.sort_values("year_month")

In [216]:
# Calls the total revenuse monthly from 2022 to 2025
colorado_revenue_monthly

,year_month,sale_amount
0,2022-01,77950.98
1,2022-02,80315.69
2,2022-03,82516.36
3,2022-04,89350.31
4,2022-05,94892.89
5,2022-06,86847.70
6,2022-07,94048.49
7,2022-08,87058.36
8,2022-09,84206.09
9,2022-10,78706.96


## Store ranking for Colorado

In [223]:
# Filters by store id and location, then ranks them by the sales amount to get the top performing store
colorado_grouped_stores = colorado_sales.groupby(["store_id", "store_location"])
colorado_store_totals = colorado_grouped_stores["sale_amount"].sum().reset_index()
colorado_store_ranking = colorado_store_totals.sort_values("sale_amount", ascending=False)

In [226]:
# Calls the ranking by total sales amount Denver is the top performing store location in Colorado with 917k in total sales
colorado_store_ranking

,store_id,store_location,sale_amount
4,705,Denver,917471.53
8,709,Greeley,640368.11
5,706,Englewood,344647.48
9,710,Lafayette,337169.61
2,703,Boulder,321197.88
15,716,Sedalia,319776.13
0,701,Aurora,317198.67
7,708,Grand Junction,306834.49
6,707,Fort Collins,302660.34
3,704,Castle Rock,299122.02


In [242]:
# Customer sales rankings, based on their rewards 
colorado_customer_info = customers[["cust_id", "name"]]
colorado_customer_sales_ranking = colorado_sales.merge(colorado_customer_info,left_on="rewardsid",right_on="cust_id",how="left")
colorado_customer_grouped = colorado_customer_sales_ranking.groupby(["cust_id", "name","rewardsid"])
colorado_customer_totals = colorado_customer_grouped["sale_amount"].sum().reset_index()
colorado_top_customers = colorado_customer_totals.sort_values("sale_amount", ascending=False)


In [244]:
# Customer sales rankings, based on their rewards Gloria Mendoza spent the most with 7.8k
colorado_top_customers

,cust_id,name,rewardsid,sale_amount
244,245,Gloria Mendoza,245,7855.66
374,375,C.J. Cregg,375,5006.57
379,380,Abbey Bartlet,380,4605.19
241,242,Lorna Morello,242,4381.15
278,279,Adrian Veidt,279,4365.77
...,...,...,...,...
435,436,Newman,436,79.04
96,97,Jane Lane,97,78.18
16,17,Mike E.,17,77.60
276,277,Laurie Blake,277,75.00


In [193]:
colorado_product_info = products[["prod_num", "categoryid"]]
colorado_product = colorado_sales.merge(colorado_product_info, on="prod_num", how="left")
colorado_grouped = colorado_product.groupby(["year_month", "categoryid"])
colorado_transactions = colorado_grouped["sale_amount"].count().reset_index()
colorado_transactions = colorado_transactions.rename(columns={"sale_amount": "transaction_count"})

In [194]:
colorado_revenue = colorado_grouped["sale_amount"].sum().reset_index()
colorado_revenue = colorado_revenue.sort_values(["year_month", "categoryid"])

## Maine Filtering

In [195]:
maine_stores = store[store["state"] == "Maine"]
maine_id = maine_stores["store_id"].unique()
maine_sales = sales[sales["store_id"].isin(maine_id)]

## Monthly total revenue for Maine

In [196]:
# making the transaction date a readable format as well as changing the name to year_month for readablity
maine_sales = maine_sales.merge(maine_stores[["store_id", "store_location"]],on="store_id",how="left")
maine_sales["transaction_date"] = pd.to_datetime(maine_sales["transaction_date"])
maine_sales["year_month"] = maine_sales["transaction_date"].dt.to_period("M")

# Adds the total revenue monthly
maine_grouped_monthly = maine_sales.groupby("year_month")
maine_revenue_monthly = maine_grouped_monthly["sale_amount"].sum().reset_index()
maine_revenue_monthly = maine_revenue_monthly.sort_values("year_month")

In [208]:
# Calls the total revenuse monthly from 2022 to 2025
maine_revenue_monthly

,year_month,sale_amount
0,2022-01,15700.31
1,2022-02,21008.29
2,2022-03,23173.23
3,2022-04,20169.19
4,2022-05,22631.11
5,2022-06,31573.93
6,2022-07,19371.11
7,2022-08,27979.07
8,2022-09,27393.30
9,2022-10,24947.09


In [198]:
maine_grouped_stores = maine_sales.groupby(["store_id", "store_location"])
maine_store_totals = maine_grouped_stores["sale_amount"].sum().reset_index()
maine_store_ranking = maine_store_totals.sort_values("sale_amount", ascending=False)

In [199]:
maine_customer_info = customers[["cust_id", "name"]]
maine_customer_sales_ranking = maine_sales.merge(maine_customer_info,left_on="rewardsid",right_on="cust_id",how="left")

In [200]:
maine_product_info = products[["prod_num", "categoryid"]]
maine_product = maine_sales.merge(maine_product_info, on="prod_num", how="left")
maine_grouped = maine_product.groupby(["year_month", "categoryid"])
maine_transactions = maine_grouped["sale_amount"].count().reset_index()
maine_transactions = maine_transactions.rename(columns={"sale_amount": "transaction_count"})

In [201]:
maine_revenue = maine_grouped["sale_amount"].sum().reset_index()
maine_revenue = maine_revenue.sort_values(["year_month", "categoryid"])

In [202]:
## Reading the files